In [1]:
%load_ext autoreload
%autoreload 2
import talib
from lark import Lark, Transformer

In [2]:


from pklib.strategy import *
from pklib.utilities import *
from pklib.pkindicators import calculate_zigzag


In [3]:

asset, quote, timeframe, exchange = 'BTC', 'USDT', '8h', 'binance'
df = load_candles('binance',asset, quote, timeframe)#['2020':'2024']#.iloc[-35000:-5000]
# df

In [4]:
import pandas as pd
import talib
from lark import Lark, Transformer, v_args
from lark.tree import Tree

grammar = """
    start: expression+

    expression: comparison
              | operation
              | indicator
              | logical
              | "(" expression ")" -> parens
              | NAME
              | NUMBER

    operation: expression "+" expression         -> add
             | expression "-" expression         -> sub
             | expression "*" expression         -> mul
             | expression "/" expression         -> div

    comparison: expression ">" expression        -> gt
              | expression "<" expression        -> lt
              | expression ">=" expression       -> gte
              | expression "<=" expression       -> lte
              | expression "==" expression       -> eq
              | expression "!=" expression       -> neq

    logical: expression "AND" expression   -> logical_and
           | expression "OR" expression    -> logical_or
           | "NOT" expression              -> logical_not

    indicator: "SMA" "(" NAME "," NUMBER ")" -> sma
             | "EMA" "(" NAME "," NUMBER ")" -> ema
             | "RSI" "(" NAME "," NUMBER ")" -> rsi
             | "MACD" "(" NAME "," NUMBER "," NUMBER "," NUMBER ")" -> macd
             | "BBANDS" "(" NAME "," NUMBER "," NUMBER "," NUMBER ")" -> bbands
             | "ATR" "(" NAME "," NUMBER ")" -> atr
             | "OBV" "(" NAME "," NAME ")" -> obv
             | "ADX" "(" NAME "," NUMBER ")" -> adx
             | "STOCH" "(" NAME "," NAME "," NUMBER "," NUMBER ")" -> stoch
             | "CCI" "(" NAME "," NUMBER ")" -> cci
             | "MFI" "(" NAME "," NAME "," NAME "," NAME "," NUMBER ")" -> mfi  // Updated MFI to accept NAMEs

    %import common.CNAME -> NAME
    %import common.NUMBER
    %import common.WS
    %ignore WS
"""



In [17]:
class QueryTransformer(Transformer):
    def __init__(self, df):
        self.df = df

    def extract_value(self, item):
        """Helper to evaluate sub-expressions recursively."""
        if isinstance(item, Tree):
            # Recursively evaluate the Tree node
            return self.transform(item)
        return item

    def add(self, items):
        return self.extract_value(items[0]) + self.extract_value(items[1])

    def sub(self, items):
        return self.extract_value(items[0]) - self.extract_value(items[1])

    def mul(self, items):
        return self.extract_value(items[0]) * self.extract_value(items[1])

    def div(self, items):
        return self.extract_value(items[0]) / self.extract_value(items[1])

    def gt(self, items):
        # Ensure both sides are evaluated before comparison
        left = self.extract_value(items[0])
        right = self.extract_value(items[1])
        print(f"Comparing {left} > {right}")
        return left > right

    def lt(self, items):
        left = self.extract_value(items[0])
        right = self.extract_value(items[1])
        return left < right

    def gte(self, items):
        left = self.extract_value(items[0])
        right = self.extract_value(items[1])
        return left >= right

    def lte(self, items):
        left = self.extract_value(items[0])
        right = self.extract_value(items[1])
        return left <= right

    def eq(self, items):
        left = self.extract_value(items[0])
        right = self.extract_value(items[1])
        return left == right

    def neq(self, items):
        left = self.extract_value(items[0])
        right = self.extract_value(items[1])
        return left != right

    def logical_and(self, items):
        return self.extract_value(items[0]) & self.extract_value(items[1])

    def logical_or(self, items):
        return self.extract_value(items[0]) | self.extract_value(items[1])

    def logical_not(self, items):
        return ~self.extract_value(items[0])

    # Technical Indicators
    def sma(self, items):
        return talib.SMA(self.df[items[0]], timeperiod=int(items[1]))

    def ema(self, items):
        return talib.EMA(self.df[items[0]], timeperiod=int(items[1]))

    def rsi(self, items):
        return talib.RSI(self.df[items[0]], timeperiod=int(items[1]))

    def macd(self, items):
        macd, macdsignal, macdhist = talib.MACD(self.df[items[0]], fastperiod=int(items[1]), slowperiod=int(items[2]), signalperiod=int(items[3]))
        return macd

    def bbands(self, items):
        upperband, middleband, lowerband = talib.BBANDS(self.df[items[0]], timeperiod=int(items[1]), nbdevup=int(items[2]), nbdevdn=int(items[3]))
        return upperband

    def atr(self, items):
        return talib.ATR(self.df['high'], self.df['low'], self.df['close'], timeperiod=int(items[1]))

    def obv(self, items):
        return talib.OBV(self.df[items[0]], self.df[items[1]])

    def adx(self, items):
        return talib.ADX(self.df['high'], self.df['low'], self.df[items[0]], timeperiod=int(items[1]))

    def stoch(self, items):
        slowk, slowd = talib.STOCH(self.df['high'], self.df['low'], self.df[items[0]], fastk_period=int(items[2]), slowk_period=int(items[3]))
        return slowk

    def cci(self, items):
        return talib.CCI(self.df['high'], self.df['low'], self.df[items[0]], timeperiod=int(items[1]))

    def mfi(self, items):
        return talib.MFI(self.df['high'], self.df['low'], self.df['close'], self.df['volume'], timeperiod=int(items[3]))

    def parens(self, items):
        return self.extract_value(items[0])

    def NUMBER(self, n):
        return float(n)

    def NAME(self, n):
        return n.value  # Return the column name as a string


In [20]:
query = """
    (SMA(close, 3) > EMA(close, 3)) AND (RSI(close, 14) > 70) OR (MFI(high, low, close, volume, 14) > 50)
"""

print(parser.parse(query).pretty())

# parsed_tree = parser.parse(query)
# transformer = QueryTransformer(df)
# result = transformer.transform(parsed_tree)

# print("Query result:\n", result)


start
  expression
    logical_and
      parens
        expression
          gt
            expression
              sma
                close
                3
            expression
              ema
                close
                3
      expression
        logical_or
          parens
            expression
              gt
                expression
                  rsi
                    close
                    14
                expression	70
          parens
            expression
              gt
                expression
                  mfi
                    high
                    low
                    close
                    volume
                    14
                expression	50

